In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import glob
import os
from scipy.stats import fisher_exact
from matplotlib_venn import venn2
from matplotlib_venn import venn3



# Interaction overlap analysis

# BIOGRID

In [ ]:
import os
import numpy as np
import pandas as pd

def run_pair_triple(protein_name_1, protein_name_2, pair_name, screen_prefix):
    # === Load HGNC
    hgnc = pd.read_csv('input_data/HGNC/hgnc_complete_set.txt',sep='\t')
    columns_to_include = ['hgnc_id', 'symbol','prev_symbol','ensembl_gene_id','alias_symbol','entrez_id']
    hgnc_dict = hgnc[columns_to_include].to_dict(orient='records')

    # BIOGRID

    biogrid = pd.read_csv('input_data/BIOGRID/BIOGRID-MV-Physical-4.4.229.tab3.txt',sep='\t')
    biogrid = biogrid[(biogrid['Organism Name Interactor A'] == 'Homo sapiens') & (biogrid['Organism Name Interactor B'] == 'Homo sapiens')]

    p1_edges = biogrid[(biogrid['Official Symbol Interactor A'] == protein_name_1) | (biogrid['Official Symbol Interactor B'] == protein_name_1)]
    p2_edges =  biogrid[(biogrid['Official Symbol Interactor A'] == protein_name_2) | (biogrid['Official Symbol Interactor B'] == protein_name_2)]
    ppi_partners = pd.concat([p1_edges, p2_edges])

    ppi_partners['SortedInteractors'] = ppi_partners.apply(lambda row: '-'.join(np.sort([row['Official Symbol Interactor A'], row['Official Symbol Interactor B']])), axis=1)
    ppi_partners.drop_duplicates(subset='SortedInteractors', keep='first', inplace=True)
    ppi_partners.drop(columns='SortedInteractors', inplace=True)
    ppi_partners= ppi_partners.reset_index(drop=True)


    # === HGNC lookup
    gene_lookup = {}
    def normalize_symbol(s): return s.strip().upper() if isinstance(s, str) else None
    for entry in hgnc_dict:
        hgnc_id = entry.get('hgnc_id', np.nan)
        ensembl_id = entry.get('ensembl_gene_id', np.nan)
        entrez_id = str(entry.get('entrez_id', 'NA')) if pd.notnull(entry.get('entrez_id')) else 'NA'
        syms = []
        s = normalize_symbol(entry.get('symbol'))
        if s: syms.append(s)
        if pd.notnull(entry.get('prev_symbol')):
            syms.extend(normalize_symbol(x) for x in entry['prev_symbol'].split('|'))
        if pd.notnull(entry.get('alias_symbol')):
            syms.extend(normalize_symbol(x) for x in entry['alias_symbol'].split('|'))
        for sym in syms:
            if sym and sym not in gene_lookup:
                gene_lookup[sym] = (hgnc_id, ensembl_id, entrez_id)

    def lookup_gene_info(gene_symbol):
        key = normalize_symbol(gene_symbol)
        return gene_lookup.get(key, (np.nan, np.nan, 'NA'))

    ppi_partners['hgnc_id_1'], ppi_partners['ensembl_gene_id_1'], ppi_partners['entrez_id_1'] = zip(*ppi_partners['Official Symbol Interactor A'].apply(lookup_gene_info))
    ppi_partners['hgnc_id_2'], ppi_partners['ensembl_gene_id_2'], ppi_partners['entrez_id_2'] = zip(*ppi_partners['Official Symbol Interactor B'].apply(lookup_gene_info))

    # === Load and annotate screen
    screen = pd.read_csv(f'input_data/2_output_with_hgnc/{screen_prefix}_{pair_name}_screen_with_hgnc.csv')

    # Ensure ensembl IDs are strings and drop NaN/empty
    valid_ids = screen['ensembl_gene_id'].fillna("").astype(str).str.strip()

    # Boolean flags for interactions, but only for non-empty IDs
    screen['Interaction_1'] = (valid_ids != "") & valid_ids.isin(ppi_partners['ensembl_gene_id_1'])
    screen['Interaction_2'] = (valid_ids != "") & valid_ids.isin(ppi_partners['ensembl_gene_id_2'])

    # Partner if either column is True
    screen['Interaction_Partners'] = (screen[['Interaction_1', 'Interaction_2']].any(axis=1)).astype(int)


    # === Your slicing logic
    specific_row_3_cell = screen[screen['Cell_line'] == '2 cell line'].index[0] - 1
    specific_row_2_cell = screen[screen['Cell_line'] == '1 cell line'].index[0] - 1
    specific_row_1_cell = screen[screen['Cell_line'] == '0 cell line'].index[0] - 1

    interacted_3_cell_line = screen.loc[:specific_row_3_cell, 'Interaction_Partners'].sum()
    total_3_cell_line = specific_row_3_cell + 1
    interacted_2_cell_line = screen.loc[:specific_row_2_cell, 'Interaction_Partners'].sum() - interacted_3_cell_line
    total_2_cell_line = specific_row_2_cell + 1 - total_3_cell_line
    interacted_1_cell_line = screen.loc[:specific_row_1_cell, 'Interaction_Partners'].sum() - interacted_2_cell_line - interacted_3_cell_line
    total_1_cell_line = specific_row_1_cell + 1 - total_2_cell_line - total_3_cell_line
    interacted_0_cell_line = screen.loc[specific_row_1_cell + 2:, 'Interaction_Partners'].sum()
    total_0_cell_line = len(screen) - (specific_row_1_cell + 1)

    counts_df = pd.DataFrame({
        ' ': [0,1,2,3],
        'Interacted': [interacted_0_cell_line, interacted_1_cell_line, interacted_2_cell_line, interacted_3_cell_line],
        'Total': [total_0_cell_line,total_1_cell_line,total_2_cell_line,total_3_cell_line]
    })

    screen['Interaction_Partners'] = screen['Interaction_Partners'].replace({0: 'Not partner', 1: 'Partner'})
    screen.drop(columns=['Interaction_1','Interaction_2'], inplace=False).to_csv(f'results/2_interaction_overlap/biogrid_results/{screen_prefix}_{pair_name}_biogrid_results.csv', index=False)
    counts_df.to_csv(f"results/2_interaction_overlap/biogrid_counts/{screen_prefix}_{pair_name}_biogrid_count.csv",index=False)

    return ppi_partners, screen, counts_df


In [3]:
ppi_partners, screen_out, counts = run_pair_triple(
    protein_name_1 = "PTEN",   # PTEN
    protein_name_2="PIK3CB",   # PIK3CB
    pair_name="PTEN_PIK3CB",          # used in filenames
    screen_prefix="dunn"              # which screen to load
)


/var/folders/cs/dzdz0ppx7lld40ztphtqg_dh0000gp/T/ipykernel_5332/3694377134.py:7: DtypeWarning: Columns (32,34,38,40,50) have mixed types. Specify dtype option on import or set low_memory=False.
  hgnc = pd.read_csv('/Users/metinyazar/Desktop/drug_resistance_ppi/database_files/hgnc/hgnc_complete_set.txt',sep='\t')
/var/folders/cs/dzdz0ppx7lld40ztphtqg_dh0000gp/T/ipykernel_5332/3694377134.py:13: DtypeWarning: Columns (1,2) have mixed types. Specify dtype option on import or set low_memory=False.
  biogrid = pd.read_csv('/Users/metinyazar/Desktop/drug_resistance_ppi/database_files/biogrid_data/BIOGRID-MV-Physical-4.4.229.tab3.txt',sep='\t')


In [ ]:
import os
import numpy as np
import pandas as pd

def run_pair_5(protein_name_1, protein_name_2,protein_name_3,protein_name_4,protein_name_5, pair_name, screen_prefix):
    # === Load HGNC
    hgnc = pd.read_csv('input_data/HGNC/hgnc_complete_set.txt',sep='\t')
    columns_to_include = ['hgnc_id', 'symbol','prev_symbol','ensembl_gene_id','alias_symbol','entrez_id']
    hgnc_dict = hgnc[columns_to_include].to_dict(orient='records')

    # BIOGRID

    biogrid = pd.read_csv('input_data/BIOGRID/BIOGRID-MV-Physical-4.4.229.tab3.txt',sep='\t')
    biogrid = biogrid[(biogrid['Organism Name Interactor A'] == 'Homo sapiens') & (biogrid['Organism Name Interactor B'] == 'Homo sapiens')]

    p1_edges = biogrid[(biogrid['Official Symbol Interactor A'] == protein_name_1) | (biogrid['Official Symbol Interactor B'] == protein_name_1)]
    p2_edges =  biogrid[(biogrid['Official Symbol Interactor A'] == protein_name_2) | (biogrid['Official Symbol Interactor B'] == protein_name_2)]
    p3_edges = biogrid[(biogrid['Official Symbol Interactor A'] == protein_name_3) | (biogrid['Official Symbol Interactor B'] == protein_name_3)]
    p4_edges = biogrid[(biogrid['Official Symbol Interactor A'] == protein_name_4) | (biogrid['Official Symbol Interactor B'] == protein_name_4)]
    p5_edges = biogrid[(biogrid['Official Symbol Interactor A'] == protein_name_5) | (biogrid['Official Symbol Interactor B'] == protein_name_5)]

    ppi_partners = pd.concat([p1_edges, p2_edges,p3_edges,p4_edges,p5_edges])

    ppi_partners['SortedInteractors'] = ppi_partners.apply(lambda row: '-'.join(np.sort([row['Official Symbol Interactor A'], row['Official Symbol Interactor B']])), axis=1)
    ppi_partners.drop_duplicates(subset='SortedInteractors', keep='first', inplace=True)
    ppi_partners.drop(columns='SortedInteractors', inplace=True)
    ppi_partners= ppi_partners.reset_index(drop=True)


    # === HGNC lookup
    gene_lookup = {}
    def normalize_symbol(s): return s.strip().upper() if isinstance(s, str) else None
    for entry in hgnc_dict:
        hgnc_id = entry.get('hgnc_id', np.nan)
        ensembl_id = entry.get('ensembl_gene_id', np.nan)
        entrez_id = str(entry.get('entrez_id', 'NA')) if pd.notnull(entry.get('entrez_id')) else 'NA'
        syms = []
        s = normalize_symbol(entry.get('symbol'))
        if s: syms.append(s)
        if pd.notnull(entry.get('prev_symbol')):
            syms.extend(normalize_symbol(x) for x in entry['prev_symbol'].split('|'))
        if pd.notnull(entry.get('alias_symbol')):
            syms.extend(normalize_symbol(x) for x in entry['alias_symbol'].split('|'))
        for sym in syms:
            if sym and sym not in gene_lookup:
                gene_lookup[sym] = (hgnc_id, ensembl_id, entrez_id)

    def lookup_gene_info(gene_symbol):
        key = normalize_symbol(gene_symbol)
        return gene_lookup.get(key, (np.nan, np.nan, 'NA'))

    ppi_partners['hgnc_id_1'], ppi_partners['ensembl_gene_id_1'], ppi_partners['entrez_id_1'] = zip(*ppi_partners['Official Symbol Interactor A'].apply(lookup_gene_info))
    ppi_partners['hgnc_id_2'], ppi_partners['ensembl_gene_id_2'], ppi_partners['entrez_id_2'] = zip(*ppi_partners['Official Symbol Interactor B'].apply(lookup_gene_info))

    # === Load and annotate screen
    screen = pd.read_csv(f'input_data/2_output_with_hgnc/{screen_prefix}_{pair_name}_screen_with_hgnc.csv')

    # Ensure ensembl IDs are strings and drop NaN/empty
    valid_ids = screen['ensembl_gene_id'].fillna("").astype(str).str.strip()

    # Boolean flags for interactions, but only for non-empty IDs
    screen['Interaction_1'] = (valid_ids != "") & valid_ids.isin(ppi_partners['ensembl_gene_id_1'])
    screen['Interaction_2'] = (valid_ids != "") & valid_ids.isin(ppi_partners['ensembl_gene_id_2'])

    # Partner if either column is True
    screen['Interaction_Partners'] = (screen[['Interaction_1', 'Interaction_2']].any(axis=1)).astype(int)


    # === Your slicing logic
    specific_row_3_cell = screen[screen['Cell_line'] == '2 cell line'].index[0] - 1
    specific_row_2_cell = screen[screen['Cell_line'] == '1 cell line'].index[0] - 1
    specific_row_1_cell = screen[screen['Cell_line'] == '0 cell line'].index[0] - 1

    interacted_3_cell_line = screen.loc[:specific_row_3_cell, 'Interaction_Partners'].sum()
    total_3_cell_line = specific_row_3_cell + 1
    interacted_2_cell_line = screen.loc[:specific_row_2_cell, 'Interaction_Partners'].sum() - interacted_3_cell_line
    total_2_cell_line = specific_row_2_cell + 1 - total_3_cell_line
    interacted_1_cell_line = screen.loc[:specific_row_1_cell, 'Interaction_Partners'].sum() - interacted_2_cell_line - interacted_3_cell_line
    total_1_cell_line = specific_row_1_cell + 1 - total_2_cell_line - total_3_cell_line
    interacted_0_cell_line = screen.loc[specific_row_1_cell + 2:, 'Interaction_Partners'].sum()
    total_0_cell_line = len(screen) - (specific_row_1_cell + 1)

    counts_df = pd.DataFrame({
        ' ': [0,1,2,3],
        'Interacted': [interacted_0_cell_line, interacted_1_cell_line, interacted_2_cell_line, interacted_3_cell_line],
        'Total': [total_0_cell_line,total_1_cell_line,total_2_cell_line,total_3_cell_line]
    })

    screen['Interaction_Partners'] = screen['Interaction_Partners'].replace({0: 'Not partner', 1: 'Partner'})
    screen.drop(columns=['Interaction_1','Interaction_2'], inplace=False).to_csv(f'results/2_interaction_overlap/biogrid_results/{screen_prefix}_{pair_name}_biogrid_results.csv', index=False)
    counts_df.to_csv(f"results/2_interaction_overlap/biogrid_counts/{screen_prefix}_{pair_name}_biogrid_count.csv",index=False)

    return ppi_partners, screen, counts_df


In [3]:
ppi_partners, screen_out, counts = run_pair_5(
    protein_name_1 = "PTEN",   # PTEN
    protein_name_2="AKT1",   # AKT1
    protein_name_3="AKT2",   # AKT2
    protein_name_4="AKT3"  , # AKT3
    protein_name_5="AKT4",  # AKT4
    pair_name="PTEN_AKT",          # used in filenames
    screen_prefix="dunn"              # which screen to load
)


/var/folders/cs/dzdz0ppx7lld40ztphtqg_dh0000gp/T/ipykernel_22276/961236139.py:7: DtypeWarning: Columns (32,34,38,40,50) have mixed types. Specify dtype option on import or set low_memory=False.
  hgnc = pd.read_csv('/Users/metinyazar/Desktop/drug_resistance_ppi/database_files/hgnc/hgnc_complete_set.txt',sep='\t')
/var/folders/cs/dzdz0ppx7lld40ztphtqg_dh0000gp/T/ipykernel_22276/961236139.py:13: DtypeWarning: Columns (1,2) have mixed types. Specify dtype option on import or set low_memory=False.
  biogrid = pd.read_csv('/Users/metinyazar/Desktop/drug_resistance_ppi/database_files/biogrid_data/BIOGRID-MV-Physical-4.4.229.tab3.txt',sep='\t')


In [ ]:
import os
import numpy as np
import pandas as pd

def run_pair_double_five_gene(protein_name_1, protein_name_2, protein_name_3,protein_name_4,protein_name_5, pair_name, screen_prefix):
    # === Load HGNC
    hgnc = pd.read_csv('input_data/HGNC/hgnc_complete_set.txt',sep='\t')
    columns_to_include = ['hgnc_id', 'symbol','prev_symbol','ensembl_gene_id','alias_symbol','entrez_id']
    hgnc_dict = hgnc[columns_to_include].to_dict(orient='records')

    # BIOGRID

    biogrid = pd.read_csv('input_data/BIOGRID/BIOGRID-MV-Physical-4.4.229.tab3.txt',sep='\t')
    biogrid = biogrid[(biogrid['Organism Name Interactor A'] == 'Homo sapiens') & (biogrid['Organism Name Interactor B'] == 'Homo sapiens')]

    p1_edges = biogrid[(biogrid['Official Symbol Interactor A'] == protein_name_1) | (biogrid['Official Symbol Interactor B'] == protein_name_1)]
    p2_edges =  biogrid[(biogrid['Official Symbol Interactor A'] == protein_name_2) | (biogrid['Official Symbol Interactor B'] == protein_name_2)]
    p3_edges = biogrid[(biogrid['Official Symbol Interactor A'] == protein_name_3) | (biogrid['Official Symbol Interactor B'] == protein_name_3)]
    p4_edges = biogrid[(biogrid['Official Symbol Interactor A'] == protein_name_4) | (biogrid['Official Symbol Interactor B'] == protein_name_4)]
    p5_edges = biogrid[(biogrid['Official Symbol Interactor A'] == protein_name_5) | (biogrid['Official Symbol Interactor B'] == protein_name_5)]

    ppi_partners = pd.concat([p1_edges, p2_edges,p3_edges,p4_edges,p5_edges])

    ppi_partners['SortedInteractors'] = ppi_partners.apply(lambda row: '-'.join(np.sort([row['Official Symbol Interactor A'], row['Official Symbol Interactor B']])), axis=1)
    ppi_partners.drop_duplicates(subset='SortedInteractors', keep='first', inplace=True)
    ppi_partners.drop(columns='SortedInteractors', inplace=True)
    ppi_partners= ppi_partners.reset_index(drop=True)


    
    # === HGNC lookup
    gene_lookup = {}
    def normalize_symbol(s): return s.strip().upper() if isinstance(s, str) else None
    for entry in hgnc_dict:
        hgnc_id = entry.get('hgnc_id', np.nan)
        ensembl_id = entry.get('ensembl_gene_id', np.nan)
        entrez_id = str(entry.get('entrez_id', 'NA')) if pd.notnull(entry.get('entrez_id')) else 'NA'
        syms = []
        s = normalize_symbol(entry.get('symbol'))
        if s: syms.append(s)
        if pd.notnull(entry.get('prev_symbol')):
            syms.extend(normalize_symbol(x) for x in entry['prev_symbol'].split('|'))
        if pd.notnull(entry.get('alias_symbol')):
            syms.extend(normalize_symbol(x) for x in entry['alias_symbol'].split('|'))
        for sym in syms:
            if sym and sym not in gene_lookup:
                gene_lookup[sym] = (hgnc_id, ensembl_id, entrez_id)

    def lookup_gene_info(gene_symbol):
        key = normalize_symbol(gene_symbol)
        return gene_lookup.get(key, (np.nan, np.nan, 'NA'))

    ppi_partners['hgnc_id_1'], ppi_partners['ensembl_gene_id_1'], ppi_partners['entrez_id_1'] = zip(*ppi_partners['Official Symbol Interactor A'].apply(lookup_gene_info))
    ppi_partners['hgnc_id_2'], ppi_partners['ensembl_gene_id_2'], ppi_partners['entrez_id_2'] = zip(*ppi_partners['Official Symbol Interactor B'].apply(lookup_gene_info))

    # === Load and annotate screen
    screen = pd.read_csv(f'input_data/2_output_with_hgnc/{screen_prefix}_{pair_name}_screen_with_hgnc.csv')
    
    # Ensure ensembl IDs are strings and drop NaN/empty
    valid_ids = screen['ensembl_gene_id'].fillna("").astype(str).str.strip()

    # Boolean flags for interactions, but only for non-empty IDs
    screen['Interaction_1'] = (valid_ids != "") & valid_ids.isin(ppi_partners['ensembl_gene_id_1'])
    screen['Interaction_2'] = (valid_ids != "") & valid_ids.isin(ppi_partners['ensembl_gene_id_2'])

    # Partner if either column is True
    screen['Interaction_Partners'] = (screen[['Interaction_1', 'Interaction_2']].any(axis=1)).astype(int)


    specific_row_2_cell = screen[screen['Cell_line'] == '1 cell line'].index[0] - 1
    specific_row_1_cell = screen[screen['Cell_line'] == '0 cell line'].index[0] - 1

    interacted_2_cell_line = screen.loc[:specific_row_2_cell, 'Interaction_Partners'].sum() 
    total_2_cell_line = specific_row_2_cell + 1 
    interacted_1_cell_line = screen.loc[:specific_row_1_cell, 'Interaction_Partners'].sum() - interacted_2_cell_line 
    total_1_cell_line = specific_row_1_cell + 1 - total_2_cell_line 

    interacted_0_cell_line = screen.loc[specific_row_1_cell + 1:, 'Interaction_Partners'].sum()
    total_0_cell_line = len(screen) - (specific_row_1_cell + 1)

    

    counts_df = pd.DataFrame({
        ' ': [0,1,2],
        'Interacted': [interacted_0_cell_line, interacted_1_cell_line, interacted_2_cell_line],
        'Total': [total_0_cell_line,total_1_cell_line,total_2_cell_line]
    })

    screen['Interaction_Partners'] = screen['Interaction_Partners'].replace({0: 'Not partner', 1: 'Partner'})
    screen.drop(columns=['Interaction_1','Interaction_2'], inplace=False).to_csv(f'results/2_interaction_overlap/biogrid_results/{screen_prefix}_{pair_name}_biogrid_results.csv', index=False)
    counts_df.to_csv(f"results/2_interaction_overlap/biogrid_counts/{screen_prefix}_{pair_name}_biogrid_count.csv",index=False)

    return ppi_partners, screen, counts_df


In [7]:
ppi_partners, screen_out, counts = run_pair_double_five_gene(
    protein_name_1 = "KRAS",   # KRAS
    protein_name_2="MAP2K1",   # MAP2K1
    protein_name_3="MEK1",   # MEK1
    protein_name_4="MAP2K2"  , # MAP2K2
    protein_name_5="MEK2",  # MEK2
    pair_name="KRAS_MEK",          # used in filenames
    screen_prefix="wang"              # which screen to load
)


/var/folders/cs/dzdz0ppx7lld40ztphtqg_dh0000gp/T/ipykernel_5332/1603602914.py:7: DtypeWarning: Columns (32,34,38,40,50) have mixed types. Specify dtype option on import or set low_memory=False.
  hgnc = pd.read_csv('/Users/metinyazar/Desktop/drug_resistance_ppi/database_files/hgnc/hgnc_complete_set.txt',sep='\t')
/var/folders/cs/dzdz0ppx7lld40ztphtqg_dh0000gp/T/ipykernel_5332/1603602914.py:13: DtypeWarning: Columns (1,2) have mixed types. Specify dtype option on import or set low_memory=False.
  biogrid = pd.read_csv('/Users/metinyazar/Desktop/drug_resistance_ppi/database_files/biogrid_data/BIOGRID-MV-Physical-4.4.229.tab3.txt',sep='\t')


In [ ]:
import os
import numpy as np
import pandas as pd

def run_pair_double_two_gene(protein_name_1, protein_name_2 ,pair_name, screen_prefix):
    # === Load HGNC
    hgnc = pd.read_csv('input_data/HGNC/hgnc_complete_set.txt',sep='\t')
    columns_to_include = ['hgnc_id', 'symbol','prev_symbol','ensembl_gene_id','alias_symbol','entrez_id']
    hgnc_dict = hgnc[columns_to_include].to_dict(orient='records')

    # BIOGRID

    biogrid = pd.read_csv('input_data/BIOGRID/BIOGRID-MV-Physical-4.4.229.tab3.txt',sep='\t')
    biogrid = biogrid[(biogrid['Organism Name Interactor A'] == 'Homo sapiens') & (biogrid['Organism Name Interactor B'] == 'Homo sapiens')]

    p1_edges = biogrid[(biogrid['Official Symbol Interactor A'] == protein_name_1) | (biogrid['Official Symbol Interactor B'] == protein_name_1)]
    p2_edges =  biogrid[(biogrid['Official Symbol Interactor A'] == protein_name_2) | (biogrid['Official Symbol Interactor B'] == protein_name_2)]
    ppi_partners = pd.concat([p1_edges, p2_edges])

    ppi_partners['SortedInteractors'] = ppi_partners.apply(lambda row: '-'.join(np.sort([row['Official Symbol Interactor A'], row['Official Symbol Interactor B']])), axis=1)
    ppi_partners.drop_duplicates(subset='SortedInteractors', keep='first', inplace=True)
    ppi_partners.drop(columns='SortedInteractors', inplace=True)
    ppi_partners= ppi_partners.reset_index(drop=True)

    # === HGNC lookup
    gene_lookup = {}

    def normalize_symbol(s): return s.strip().upper() if isinstance(s, str) else None
    for entry in hgnc_dict:
        hgnc_id = entry.get('hgnc_id', np.nan)
        ensembl_id = entry.get('ensembl_gene_id', np.nan)
        entrez_id = str(entry.get('entrez_id', 'NA')) if pd.notnull(entry.get('entrez_id')) else 'NA'
        syms = []
        s = normalize_symbol(entry.get('symbol'))
        if s: syms.append(s)
        if pd.notnull(entry.get('prev_symbol')):
            syms.extend(normalize_symbol(x) for x in entry['prev_symbol'].split('|'))
        if pd.notnull(entry.get('alias_symbol')):
            syms.extend(normalize_symbol(x) for x in entry['alias_symbol'].split('|'))
        for sym in syms:
            if sym and sym not in gene_lookup:
                gene_lookup[sym] = (hgnc_id, ensembl_id, entrez_id)

    def lookup_gene_info(gene_symbol):
        key = normalize_symbol(gene_symbol)
        return gene_lookup.get(key, (np.nan, np.nan, 'NA'))

    ppi_partners['hgnc_id_1'], ppi_partners['ensembl_gene_id_1'], ppi_partners['entrez_id_1'] = zip(*ppi_partners['Official Symbol Interactor A'].apply(lookup_gene_info))
    ppi_partners['hgnc_id_2'], ppi_partners['ensembl_gene_id_2'], ppi_partners['entrez_id_2'] = zip(*ppi_partners['Official Symbol Interactor B'].apply(lookup_gene_info))

    # === Load and annotate screen
    screen = pd.read_csv(f'input_data/2_output_with_hgnc/{screen_prefix}_{pair_name}_screen_with_hgnc.csv')

    # Ensure ensembl IDs are strings and drop NaN/empty
    valid_ids = screen['ensembl_gene_id'].fillna("").astype(str).str.strip()

    # Boolean flags for interactions, but only for non-empty IDs
    screen['Interaction_1'] = (valid_ids != "") & valid_ids.isin(ppi_partners['ensembl_gene_id_1'])
    screen['Interaction_2'] = (valid_ids != "") & valid_ids.isin(ppi_partners['ensembl_gene_id_2'])

    # Partner if either column is True
    screen['Interaction_Partners'] = (screen[['Interaction_1', 'Interaction_2']].any(axis=1)).astype(int)


    specific_row_2_cell = screen[screen['Cell_line'] == '1 cell line'].index[0] - 1
    specific_row_1_cell = screen[screen['Cell_line'] == '0 cell line'].index[0] - 1

    interacted_2_cell_line = screen.loc[:specific_row_2_cell, 'Interaction_Partners'].sum() 
    total_2_cell_line = specific_row_2_cell + 1 
    interacted_1_cell_line = screen.loc[:specific_row_1_cell, 'Interaction_Partners'].sum() - interacted_2_cell_line 
    total_1_cell_line = specific_row_1_cell + 1 - total_2_cell_line 

    interacted_0_cell_line = screen.loc[specific_row_1_cell + 1:, 'Interaction_Partners'].sum()
    total_0_cell_line = len(screen) - (specific_row_1_cell + 1)

    

    counts_df = pd.DataFrame({
        ' ': [0,1,2],
        'Interacted': [interacted_0_cell_line, interacted_1_cell_line, interacted_2_cell_line],
        'Total': [total_0_cell_line,total_1_cell_line,total_2_cell_line]
    })

    screen['Interaction_Partners'] = screen['Interaction_Partners'].replace({0: 'Not partner', 1: 'Partner'})
    screen.drop(columns=['Interaction_1','Interaction_2'], inplace=False).to_csv(f'results/2_interaction_overlap/biogrid_results/{screen_prefix}_{pair_name}_biogrid_results.csv', index=False)
    counts_df.to_csv(f"results/2_interaction_overlap/biogrid_counts/{screen_prefix}_{pair_name}_biogrid_count.csv",index=False)

    return ppi_partners, screen, counts_df


In [9]:
ppi_partners, screen_out, counts = run_pair_double_two_gene(
    protein_name_1="BRCA1",   # BRCA1
    protein_name_2="PARP1",   # PARP1
    pair_name="BRCA1_PARP1",          
    screen_prefix="noordermeer")    

/var/folders/cs/dzdz0ppx7lld40ztphtqg_dh0000gp/T/ipykernel_5332/369582323.py:7: DtypeWarning: Columns (32,34,38,40,50) have mixed types. Specify dtype option on import or set low_memory=False.
  hgnc = pd.read_csv('/Users/metinyazar/Desktop/drug_resistance_ppi/database_files/hgnc/hgnc_complete_set.txt',sep='\t')
/var/folders/cs/dzdz0ppx7lld40ztphtqg_dh0000gp/T/ipykernel_5332/369582323.py:13: DtypeWarning: Columns (1,2) have mixed types. Specify dtype option on import or set low_memory=False.
  biogrid = pd.read_csv('/Users/metinyazar/Desktop/drug_resistance_ppi/database_files/biogrid_data/BIOGRID-MV-Physical-4.4.229.tab3.txt',sep='\t')


In [ ]:
import os
import numpy as np
import pandas as pd

def run_pair_two_gene_agg(protein_name_1, protein_name_2, pair_name, screen_prefix):
    # === Load HGNC
    hgnc = pd.read_csv('input_data/HGNC/hgnc_complete_set.txt',sep='\t')
    columns_to_include = ['hgnc_id', 'symbol','prev_symbol','ensembl_gene_id','alias_symbol','entrez_id']
    hgnc_dict = hgnc[columns_to_include].to_dict(orient='records')

    # BIOGRID

    biogrid = pd.read_csv('input_data/BIOGRID/BIOGRID-MV-Physical-4.4.229.tab3.txt',sep='\t')
    biogrid = biogrid[(biogrid['Organism Name Interactor A'] == 'Homo sapiens') & (biogrid['Organism Name Interactor B'] == 'Homo sapiens')]

    p1_edges = biogrid[(biogrid['Official Symbol Interactor A'] == protein_name_1) | (biogrid['Official Symbol Interactor B'] == protein_name_1)]
    p2_edges =  biogrid[(biogrid['Official Symbol Interactor A'] == protein_name_2) | (biogrid['Official Symbol Interactor B'] == protein_name_2)]
    ppi_partners = pd.concat([p1_edges, p2_edges])

    ppi_partners['SortedInteractors'] = ppi_partners.apply(lambda row: '-'.join(np.sort([row['Official Symbol Interactor A'], row['Official Symbol Interactor B']])), axis=1)
    ppi_partners.drop_duplicates(subset='SortedInteractors', keep='first', inplace=True)
    ppi_partners.drop(columns='SortedInteractors', inplace=True)
    ppi_partners= ppi_partners.reset_index(drop=True)

    # === HGNC lookup
    gene_lookup = {}
    def normalize_symbol(s): return s.strip().upper() if isinstance(s, str) else None
    for entry in hgnc_dict:
        hgnc_id = entry.get('hgnc_id', np.nan)
        ensembl_id = entry.get('ensembl_gene_id', np.nan)
        entrez_id = str(entry.get('entrez_id', 'NA')) if pd.notnull(entry.get('entrez_id')) else 'NA'
        syms = []
        s = normalize_symbol(entry.get('symbol'))
        if s: syms.append(s)
        if pd.notnull(entry.get('prev_symbol')):
            syms.extend(normalize_symbol(x) for x in entry['prev_symbol'].split('|'))
        if pd.notnull(entry.get('alias_symbol')):
            syms.extend(normalize_symbol(x) for x in entry['alias_symbol'].split('|'))
        for sym in syms:
            if sym and sym not in gene_lookup:
                gene_lookup[sym] = (hgnc_id, ensembl_id, entrez_id)

    def lookup_gene_info(gene_symbol):
        key = normalize_symbol(gene_symbol)
        return gene_lookup.get(key, (np.nan, np.nan, 'NA'))

    ppi_partners['hgnc_id_1'], ppi_partners['ensembl_gene_id_1'], ppi_partners['entrez_id_1'] = zip(*ppi_partners['Official Symbol Interactor A'].apply(lookup_gene_info))
    ppi_partners['hgnc_id_2'], ppi_partners['ensembl_gene_id_2'], ppi_partners['entrez_id_2'] = zip(*ppi_partners['Official Symbol Interactor B'].apply(lookup_gene_info))

    # === Load and annotate screen
    screen = pd.read_csv(f'input_data/2_output_with_hgnc/{pair_name}_{screen_prefix}_aggregated.csv')

    # Ensure ensembl IDs are strings and drop NaN/empty
    valid_ids = screen['ensembl_gene_id'].fillna("").astype(str).str.strip()

    # Boolean flags for interactions, but only for non-empty IDs
    screen['Interaction_1'] = (valid_ids != "") & valid_ids.isin(ppi_partners['ensembl_gene_id_1'])
    screen['Interaction_2'] = (valid_ids != "") & valid_ids.isin(ppi_partners['ensembl_gene_id_2'])

    # Partner if either column is True
    screen['Interaction_Partners'] = (screen[['Interaction_1', 'Interaction_2']].any(axis=1)).astype(int)


    specific_row = (screen["Resistance"] == "Resistant").sum()


    interacted_0_cell_line = screen.loc[specific_row + 2:, 'Interaction_Partners'].sum()
    total_0_cell_line = len(screen) - specific_row
    interacted_1_cell_line = screen.loc[:specific_row, 'Interaction_Partners'].sum()
    total_1_cell_line = specific_row


    counts_df = pd.DataFrame({
        ' ': [0,1],
        'Interacted': [interacted_0_cell_line,interacted_1_cell_line],
        'Total': [total_0_cell_line,total_1_cell_line]})

    screen['Interaction_Partners'] = screen['Interaction_Partners'].replace({0: 'Not partner', 1: 'Partner'})
    screen.drop(columns=['Interaction_1','Interaction_2'], inplace=False).to_csv(f'results/2_interaction_overlap/biogrid_results/{pair_name}_{screen_prefix}_aggregated_biogrid_results.csv', index=False)
    counts_df.to_csv(f"results/2_interaction_overlap/biogrid_counts/{pair_name}_{screen_prefix}_aggregated_biogrid_count.csv",index=False)

    return ppi_partners, screen, counts_df


In [11]:
ppi_partners, screen_out, counts = run_pair_two_gene_agg(
    protein_name_1="ATR",   # ATR
    protein_name_2="ARID1A",   # ARID1A
    pair_name="ARID1A_ATR",          
    screen_prefix="Awwad_Llorca")          

/var/folders/cs/dzdz0ppx7lld40ztphtqg_dh0000gp/T/ipykernel_5332/3597948414.py:7: DtypeWarning: Columns (32,34,38,40,50) have mixed types. Specify dtype option on import or set low_memory=False.
  hgnc = pd.read_csv('/Users/metinyazar/Desktop/drug_resistance_ppi/database_files/hgnc/hgnc_complete_set.txt',sep='\t')
/var/folders/cs/dzdz0ppx7lld40ztphtqg_dh0000gp/T/ipykernel_5332/3597948414.py:13: DtypeWarning: Columns (1,2) have mixed types. Specify dtype option on import or set low_memory=False.
  biogrid = pd.read_csv('/Users/metinyazar/Desktop/drug_resistance_ppi/database_files/biogrid_data/BIOGRID-MV-Physical-4.4.229.tab3.txt',sep='\t')


In [12]:
ppi_partners, screen_out, counts = run_pair_two_gene_agg(
    protein_name_1="BRCA1",   # BRCA1
    protein_name_2="PARP1",   # PARP1
    pair_name="BRCA1_PARP1",          
    screen_prefix="Noordermeer_Zimmermann")    

/var/folders/cs/dzdz0ppx7lld40ztphtqg_dh0000gp/T/ipykernel_5332/3597948414.py:7: DtypeWarning: Columns (32,34,38,40,50) have mixed types. Specify dtype option on import or set low_memory=False.
  hgnc = pd.read_csv('/Users/metinyazar/Desktop/drug_resistance_ppi/database_files/hgnc/hgnc_complete_set.txt',sep='\t')
/var/folders/cs/dzdz0ppx7lld40ztphtqg_dh0000gp/T/ipykernel_5332/3597948414.py:13: DtypeWarning: Columns (1,2) have mixed types. Specify dtype option on import or set low_memory=False.
  biogrid = pd.read_csv('/Users/metinyazar/Desktop/drug_resistance_ppi/database_files/biogrid_data/BIOGRID-MV-Physical-4.4.229.tab3.txt',sep='\t')


In [ ]:
import os
import numpy as np
import pandas as pd

def run_pair_five_gene_agg(protein_name_1, protein_name_2,protein_name_3,protein_name_4,protein_name_5,pair_name, screen_prefix):
    # === Load HGNC
    hgnc = pd.read_csv('input_data/HGNC/hgnc_complete_set.txt',sep='\t')
    columns_to_include = ['hgnc_id', 'symbol','prev_symbol','ensembl_gene_id','alias_symbol','entrez_id']
    hgnc_dict = hgnc[columns_to_include].to_dict(orient='records')

    # BIOGRID

    biogrid = pd.read_csv('input_data/BIOGRID/BIOGRID-MV-Physical-4.4.229.tab3.txt',sep='\t')
    biogrid = biogrid[(biogrid['Organism Name Interactor A'] == 'Homo sapiens') & (biogrid['Organism Name Interactor B'] == 'Homo sapiens')]

    p1_edges = biogrid[(biogrid['Official Symbol Interactor A'] == protein_name_1) | (biogrid['Official Symbol Interactor B'] == protein_name_1)]
    p2_edges =  biogrid[(biogrid['Official Symbol Interactor A'] == protein_name_2) | (biogrid['Official Symbol Interactor B'] == protein_name_2)]
    p3_edges = biogrid[(biogrid['Official Symbol Interactor A'] == protein_name_3) | (biogrid['Official Symbol Interactor B'] == protein_name_3)]
    p4_edges = biogrid[(biogrid['Official Symbol Interactor A'] == protein_name_4) | (biogrid['Official Symbol Interactor B'] == protein_name_4)]
    p5_edges = biogrid[(biogrid['Official Symbol Interactor A'] == protein_name_5) | (biogrid['Official Symbol Interactor B'] == protein_name_5)]

    ppi_partners = pd.concat([p1_edges, p2_edges,p3_edges,p4_edges,p5_edges])

    ppi_partners['SortedInteractors'] = ppi_partners.apply(lambda row: '-'.join(np.sort([row['Official Symbol Interactor A'], row['Official Symbol Interactor B']])), axis=1)
    ppi_partners.drop_duplicates(subset='SortedInteractors', keep='first', inplace=True)
    ppi_partners.drop(columns='SortedInteractors', inplace=True)
    ppi_partners= ppi_partners.reset_index(drop=True)

    # === HGNC lookup
    gene_lookup = {}
    def normalize_symbol(s): return s.strip().upper() if isinstance(s, str) else None
    for entry in hgnc_dict:
        hgnc_id = entry.get('hgnc_id', np.nan)
        ensembl_id = entry.get('ensembl_gene_id', np.nan)
        entrez_id = str(entry.get('entrez_id', 'NA')) if pd.notnull(entry.get('entrez_id')) else 'NA'
        syms = []
        s = normalize_symbol(entry.get('symbol'))
        if s: syms.append(s)
        if pd.notnull(entry.get('prev_symbol')):
            syms.extend(normalize_symbol(x) for x in entry['prev_symbol'].split('|'))
        if pd.notnull(entry.get('alias_symbol')):
            syms.extend(normalize_symbol(x) for x in entry['alias_symbol'].split('|'))
        for sym in syms:
            if sym and sym not in gene_lookup:
                gene_lookup[sym] = (hgnc_id, ensembl_id, entrez_id)

    def lookup_gene_info(gene_symbol):
        key = normalize_symbol(gene_symbol)
        return gene_lookup.get(key, (np.nan, np.nan, 'NA'))

    ppi_partners['hgnc_id_1'], ppi_partners['ensembl_gene_id_1'], ppi_partners['entrez_id_1'] = zip(*ppi_partners['Official Symbol Interactor A'].apply(lookup_gene_info))
    ppi_partners['hgnc_id_2'], ppi_partners['ensembl_gene_id_2'], ppi_partners['entrez_id_2'] = zip(*ppi_partners['Official Symbol Interactor B'].apply(lookup_gene_info))

    # === Load and annotate screen
    screen = pd.read_csv(f'input_data/2_output_with_hgnc/{pair_name}_{screen_prefix}_aggregated.csv')

    # Ensure ensembl IDs are strings and drop NaN/empty
    valid_ids = screen['ensembl_gene_id'].fillna("").astype(str).str.strip()

    # Boolean flags for interactions, but only for non-empty IDs
    screen['Interaction_1'] = (valid_ids != "") & valid_ids.isin(ppi_partners['ensembl_gene_id_1'])
    screen['Interaction_2'] = (valid_ids != "") & valid_ids.isin(ppi_partners['ensembl_gene_id_2'])

    # Partner if either column is True
    screen['Interaction_Partners'] = (screen[['Interaction_1', 'Interaction_2']].any(axis=1)).astype(int)


    specific_row = (screen["Resistance"] == "Resistant").sum()


    interacted_0_cell_line = screen.loc[specific_row + 2:, 'Interaction_Partners'].sum()
    total_0_cell_line = len(screen) - specific_row
    interacted_1_cell_line = screen.loc[:specific_row, 'Interaction_Partners'].sum()
    total_1_cell_line = specific_row


    counts_df = pd.DataFrame({
        ' ': [0,1],
        'Interacted': [interacted_0_cell_line,interacted_1_cell_line],
        'Total': [total_0_cell_line,total_1_cell_line]})

    screen['Interaction_Partners'] = screen['Interaction_Partners'].replace({0: 'Not partner', 1: 'Partner'})
    screen.drop(columns=['Interaction_1','Interaction_2'], inplace=False).to_csv(f'results/2_interaction_overlap/biogrid_results/{pair_name}_{screen_prefix}_aggregated_biogrid_results.csv', index=False)
    counts_df.to_csv(f"results/2_interaction_overlap/biogrid_counts/{pair_name}_{screen_prefix}_aggregated_biogrid_count.csv",index=False)

    return ppi_partners, screen, counts_df


In [14]:
ppi_partners, screen_out, counts = run_pair_five_gene_agg(
    protein_name_1 = "KRAS",   # KRAS
    protein_name_2="MAP2K1",   # MAP2K1
    protein_name_3="MEK1",   # MEK1
    protein_name_4="MAP2K2"  , # MAP2K2
    protein_name_5="MEK2",  # MEK2
    pair_name="KRAS_MEK",          # used in filenames
    screen_prefix="Krall_Wang_Yu"              # which screen to load
)

/var/folders/cs/dzdz0ppx7lld40ztphtqg_dh0000gp/T/ipykernel_5332/1941237872.py:7: DtypeWarning: Columns (32,34,38,40,50) have mixed types. Specify dtype option on import or set low_memory=False.
  hgnc = pd.read_csv('/Users/metinyazar/Desktop/drug_resistance_ppi/database_files/hgnc/hgnc_complete_set.txt',sep='\t')
/var/folders/cs/dzdz0ppx7lld40ztphtqg_dh0000gp/T/ipykernel_5332/1941237872.py:13: DtypeWarning: Columns (1,2) have mixed types. Specify dtype option on import or set low_memory=False.
  biogrid = pd.read_csv('/Users/metinyazar/Desktop/drug_resistance_ppi/database_files/biogrid_data/BIOGRID-MV-Physical-4.4.229.tab3.txt',sep='\t')


In [5]:
ppi_partners, screen_out, counts = run_pair_five_gene_agg(
    protein_name_1 = "NRAS",   # KRAS
    protein_name_2="MAP2K1",   # MAP2K1
    protein_name_3="MEK1",   # MEK1
    protein_name_4="MAP2K2"  , # MAP2K2
    protein_name_5="MEK2",  # MEK2
    pair_name="NRAS_MEK",          # used in filenames
    screen_prefix="Hayes_Krall"              # which screen to load
)

/var/folders/cs/dzdz0ppx7lld40ztphtqg_dh0000gp/T/ipykernel_22276/1941237872.py:7: DtypeWarning: Columns (32,34,38,40,50) have mixed types. Specify dtype option on import or set low_memory=False.
  hgnc = pd.read_csv('/Users/metinyazar/Desktop/drug_resistance_ppi/database_files/hgnc/hgnc_complete_set.txt',sep='\t')
/var/folders/cs/dzdz0ppx7lld40ztphtqg_dh0000gp/T/ipykernel_22276/1941237872.py:13: DtypeWarning: Columns (1,2) have mixed types. Specify dtype option on import or set low_memory=False.
  biogrid = pd.read_csv('/Users/metinyazar/Desktop/drug_resistance_ppi/database_files/biogrid_data/BIOGRID-MV-Physical-4.4.229.tab3.txt',sep='\t')


In [ ]:
import os
import numpy as np
import pandas as pd

def run_pair_five_gene(protein_name_1, protein_name_2,protein_name_3,protein_name_4,protein_name_5,pair_name, screen_prefix):
    # === Load HGNC
    hgnc = pd.read_csv('input_data/HGNC/hgnc_complete_set.txt',sep='\t')
    columns_to_include = ['hgnc_id', 'symbol','prev_symbol','ensembl_gene_id','alias_symbol','entrez_id']
    hgnc_dict = hgnc[columns_to_include].to_dict(orient='records')

    # BIOGRID

    biogrid = pd.read_csv('input_data/BIOGRID/BIOGRID-MV-Physical-4.4.229.tab3.txt',sep='\t')
    biogrid = biogrid[(biogrid['Organism Name Interactor A'] == 'Homo sapiens') & (biogrid['Organism Name Interactor B'] == 'Homo sapiens')]

    p1_edges = biogrid[(biogrid['Official Symbol Interactor A'] == protein_name_1) | (biogrid['Official Symbol Interactor B'] == protein_name_1)]
    p2_edges =  biogrid[(biogrid['Official Symbol Interactor A'] == protein_name_2) | (biogrid['Official Symbol Interactor B'] == protein_name_2)]
    p3_edges = biogrid[(biogrid['Official Symbol Interactor A'] == protein_name_3) | (biogrid['Official Symbol Interactor B'] == protein_name_3)]
    p4_edges = biogrid[(biogrid['Official Symbol Interactor A'] == protein_name_4) | (biogrid['Official Symbol Interactor B'] == protein_name_4)]
    p5_edges = biogrid[(biogrid['Official Symbol Interactor A'] == protein_name_5) | (biogrid['Official Symbol Interactor B'] == protein_name_5)]

    ppi_partners = pd.concat([p1_edges, p2_edges,p3_edges,p4_edges,p5_edges])

    ppi_partners['SortedInteractors'] = ppi_partners.apply(lambda row: '-'.join(np.sort([row['Official Symbol Interactor A'], row['Official Symbol Interactor B']])), axis=1)
    ppi_partners.drop_duplicates(subset='SortedInteractors', keep='first', inplace=True)
    ppi_partners.drop(columns='SortedInteractors', inplace=True)
    ppi_partners= ppi_partners.reset_index(drop=True)

    # === HGNC lookup
    gene_lookup = {}
    def normalize_symbol(s): return s.strip().upper() if isinstance(s, str) else None
    for entry in hgnc_dict:
        hgnc_id = entry.get('hgnc_id', np.nan)
        ensembl_id = entry.get('ensembl_gene_id', np.nan)
        entrez_id = str(entry.get('entrez_id', 'NA')) if pd.notnull(entry.get('entrez_id')) else 'NA'
        syms = []
        s = normalize_symbol(entry.get('symbol'))
        if s: syms.append(s)
        if pd.notnull(entry.get('prev_symbol')):
            syms.extend(normalize_symbol(x) for x in entry['prev_symbol'].split('|'))
        if pd.notnull(entry.get('alias_symbol')):
            syms.extend(normalize_symbol(x) for x in entry['alias_symbol'].split('|'))
        for sym in syms:
            if sym and sym not in gene_lookup:
                gene_lookup[sym] = (hgnc_id, ensembl_id, entrez_id)

    def lookup_gene_info(gene_symbol):
        key = normalize_symbol(gene_symbol)
        return gene_lookup.get(key, (np.nan, np.nan, 'NA'))

    ppi_partners['hgnc_id_1'], ppi_partners['ensembl_gene_id_1'], ppi_partners['entrez_id_1'] = zip(*ppi_partners['Official Symbol Interactor A'].apply(lookup_gene_info))
    ppi_partners['hgnc_id_2'], ppi_partners['ensembl_gene_id_2'], ppi_partners['entrez_id_2'] = zip(*ppi_partners['Official Symbol Interactor B'].apply(lookup_gene_info))

    # === Load and annotate screen
    screen = pd.read_csv(f'input_data/2_output_with_hgnc/{screen_prefix}_{pair_name}_screen_with_hgnc.csv')

    # Ensure ensembl IDs are strings and drop NaN/empty
    valid_ids = screen['ensembl_gene_id'].fillna("").astype(str).str.strip()

    # Boolean flags for interactions, but only for non-empty IDs
    screen['Interaction_1'] = (valid_ids != "") & valid_ids.isin(ppi_partners['ensembl_gene_id_1'])
    screen['Interaction_2'] = (valid_ids != "") & valid_ids.isin(ppi_partners['ensembl_gene_id_2'])

    # Partner if either column is True
    screen['Interaction_Partners'] = (screen[['Interaction_1', 'Interaction_2']].any(axis=1)).astype(int)


    specific_row = (screen["Resistance"] == "Resistant").sum()


    interacted_0_cell_line = screen.loc[specific_row + 2:, 'Interaction_Partners'].sum()
    total_0_cell_line = len(screen) - specific_row
    interacted_1_cell_line = screen.loc[:specific_row, 'Interaction_Partners'].sum()
    total_1_cell_line = specific_row


    counts_df = pd.DataFrame({
        ' ': [0,1],
        'Interacted': [interacted_0_cell_line,interacted_1_cell_line],
        'Total': [total_0_cell_line,total_1_cell_line]})

    screen['Interaction_Partners'] = screen['Interaction_Partners'].replace({0: 'Not partner', 1: 'Partner'})
    screen.drop(columns=['Interaction_1','Interaction_2'], inplace=False).to_csv(f'results/2_interaction_overlap/biogrid_results/{screen_prefix}_{pair_name}_biogrid_results.csv', index=False)
    counts_df.to_csv(f"results/2_interaction_overlap/biogrid_counts/{screen_prefix}_{pair_name}_biogrid_count.csv",index=False)

    return ppi_partners, screen, counts_df


In [7]:
ppi_partners, screen_out, counts = run_pair_five_gene(
    protein_name_1 = "NRAS",   
    protein_name_2="CDK4",   
    protein_name_3="Cdk4",   
    protein_name_4="CDK6", 
    protein_name_5="Cdk6",  
    pair_name="NRAS_CDK4_6",          # used in filenames
    screen_prefix="hayes"              # which screen to load
)

/var/folders/cs/dzdz0ppx7lld40ztphtqg_dh0000gp/T/ipykernel_22276/459652955.py:7: DtypeWarning: Columns (32,34,38,40,50) have mixed types. Specify dtype option on import or set low_memory=False.
  hgnc = pd.read_csv('/Users/metinyazar/Desktop/drug_resistance_ppi/database_files/hgnc/hgnc_complete_set.txt',sep='\t')
/var/folders/cs/dzdz0ppx7lld40ztphtqg_dh0000gp/T/ipykernel_22276/459652955.py:13: DtypeWarning: Columns (1,2) have mixed types. Specify dtype option on import or set low_memory=False.
  biogrid = pd.read_csv('/Users/metinyazar/Desktop/drug_resistance_ppi/database_files/biogrid_data/BIOGRID-MV-Physical-4.4.229.tab3.txt',sep='\t')


In [18]:
ppi_partners, screen_out, counts = run_pair_five_gene(
    protein_name_1 = "BRAF",   # BRAF
    protein_name_2="MAP2K1",   # MAP2K1
    protein_name_3="MEK1",   # MEK1
    protein_name_4="MAP2K2"  , # MAP2K2
    protein_name_5="MEK2",  # MEK2
    pair_name="BRAF_MEK",          # used in filenames
    screen_prefix="krall"              # which screen to load
)

/var/folders/cs/dzdz0ppx7lld40ztphtqg_dh0000gp/T/ipykernel_5332/459652955.py:7: DtypeWarning: Columns (32,34,38,40,50) have mixed types. Specify dtype option on import or set low_memory=False.
  hgnc = pd.read_csv('/Users/metinyazar/Desktop/drug_resistance_ppi/database_files/hgnc/hgnc_complete_set.txt',sep='\t')
/var/folders/cs/dzdz0ppx7lld40ztphtqg_dh0000gp/T/ipykernel_5332/459652955.py:13: DtypeWarning: Columns (1,2) have mixed types. Specify dtype option on import or set low_memory=False.
  biogrid = pd.read_csv('/Users/metinyazar/Desktop/drug_resistance_ppi/database_files/biogrid_data/BIOGRID-MV-Physical-4.4.229.tab3.txt',sep='\t')
